In [1]:
pip install konlpy

   ---------------------------------------- 0.0/19.4 MB ? eta -:--:--
   ------------------------------------- -- 18.4/19.4 MB 92.1 MB/s eta 0:00:01
   ---------------------------------------- 19.4/19.4 MB 81.6 MB/s eta 0:00:00

   -------------------- ------------------- 1/2 [konlpy]
   ---------------------------------------- 2/2 [konlpy]

Note: you may need to restart the kernel to use updated packages.


In [9]:
#컬리에서 수집한 데이터 엑셀 파일로 저장하기
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from openpyxl import Workbook
from konlpy.tag import Okt#한글을 자연어로 인식해서 편집가능하게 해주는 
from collections import Counter#리스트 형태에 들어가 있는 문자에 대한 집계
import time

options = Options()
options.add_argument("--no-sandbox")#구글 크롬 보안정책에 태클걸지마라
options.add_argument("--disable-dev-shm-usage")#브라우저 메모리공간 사용하지 말고 로컬컴퓨터 메모리를 사용해라
options.add_argument("--headless")#브라우저 슬립 패스

driver = webdriver.Chrome(options=options)

url = "https://www.kurly.com/goods/5030559"
test = driver.get(url)
time.sleep(5)
# HTML 가져오기
soup = BeautifulSoup(driver.page_source, "html.parser")
# 리뷰 텍스트 출력하기
reviews = soup.select("p.css-y49dcn.e36z05c13")
review_lists = [review.get_text().strip() for review in reviews]

driver.quit()

wb = Workbook()
ws = wb.active
ws.title = "Kurly Reviews"
ws.append(["리뷰 번호", "리뷰내용"])

for index, item in enumerate(review_lists) :
    ws.append([index+1, item])

okt = Okt()
all_text = " ".join(review_lists)#데이터 추출하기 위해 하나의 문자열로 변환
nouns = okt.nouns(all_text)
filtered_nouns = [n for n in nouns if len(n) > 1]#두글자 이상인 값만 가져와서 filtered_nouns 리스트에 저장
count = Counter(filtered_nouns)#리스트의 데이터에 각 데이터가 몇번 있는지 추출

ws_keywords = wb.create_sheet("Keyword Frequency")#새로운 시트 생성
ws_keywords.append(["키워드", "빈도수"])#새로운 시트에 키워드, 빈도수 열
for word, freq in count.most_common(30) :
    ws_keywords.append([word, freq])

wb.save("Kurly_reviews(food1).xlsx")
print("엑셀 저장 완료 : Kurly_reviews(food1).xlsx")

엑셀 저장 완료 : Kurly_reviews(food1).xlsx


In [34]:
#키워드 수집한 엑셀에서 데이터 추출해보기
from openpyxl import load_workbook
from konlpy.tag import Okt
from collections import Counter

wb = load_workbook("연관키워드 20250722 링귀걸이.xlsx")
ws = wb.active

texts = [row[0] for row in ws.iter_rows(min_row = 2, max_col = 1, values_only=True) if row[0]] #각각의 열중 두번째 행부터 최대 첫번째 열에 값이 있는것을 찾아와서 리스트 형태로 texts 변수 안에 넣어라

okt = Okt()
all_text = " ".join(texts)
nouns = okt.nouns(all_text)
filtered_nouns = [n for n in nouns if len(n) > 1]
count = Counter(filtered_nouns)

ws_keywords = wb.create_sheet("키워드 전체분석")
ws_keywords.append(["명사", "빈도수"])
for word, freq in sorted(count.items(), key = lambda x: x[1], reverse = True) :#lambda => 익명함수, reverse => 내림차순
    ws_keywords.append([word, freq])

wb.save("연관키워드_분석결과(링귀걸이).xlsx")
print("엑셀저장 완료")
print(count)

엑셀저장 완료
Counter({'귀걸이': 462, '터치': 35, '명품': 35, '실버': 26, '여자': 24, '브랜드': 23, '드롭': 19, '추천': 19, '큐빅': 17, '골드': 16, '목걸이': 15, '여성': 14, '볼드': 13, '하트': 12, '일리': 11, '주얼리': 11, '악세사리': 10, '진주': 10, '이어링': 10, '미니': 8, '쇼핑몰': 7, '귀고리': 6, '선물': 5, '심플': 5, '물방울': 5, '클립': 5, '리본': 4, '화이트골드': 4, '꼬임': 4, '가격': 4, '쥬얼리': 4, '블랙': 4, '커플링': 4, '피어싱': 3, '금침': 3, '연예인': 3, '모양': 3, '남자': 3, '귓바퀴': 3, '엄마': 3, '클로버': 3, '나비': 3, '볼륨': 3, '유행': 3, '체인': 3, '보석': 3, '반지': 3, '니트': 3, '바지': 3, '등산': 3, '후드': 3, '팔찌': 3, '헤어': 3, '패션': 2, '도금': 2, '사이트': 2, '펜던트': 2, '팬던트': 2, '파이프': 2, '케이': 2, '탄생석': 2, '여자친구': 2, '알러지': 2, '여름': 2, '루비': 2, '종로': 2, '스터드': 2, '편집': 2, '액세서리': 2, '진주원': 2, '하객': 2, '자개': 2, '한쪽': 2, '디올': 2, '커프': 2, '당일': 2, '배송': 2, '비즈': 2, '스타일': 2, '유색': 2, '무광': 2, '유니크': 2, '가방': 2, '남성': 2, '자켓': 2, '패딩': 2, '리스트': 2, '위드': 2, '하이': 2, '앤디': 1, '금링': 1, '디자이너': 1, '메이커': 1, '대학생': 1, '빅링': 1, '이드': 1, '왕관': 1, '정유미': 1, '웨이': 1, '판매': 1, '금볼': 1, '플라워': 1, '별자리'

In [37]:
#키워드 수집한 엑셀에서 데이터 추출해보기
from openpyxl import load_workbook
from konlpy.tag import Okt
from collections import Counter

wb = load_workbook("Keyword Stats 2025-07-22(청바지구글).xlsx")
ws = wb.active

texts = [row[0] for row in ws.iter_rows(min_row=2, max_col=1, values_only=True) if row[0]]

okt = Okt()
all_text = " ".join(texts)
nouns = okt.nouns(all_text)
filtered_nouns = [n for n in nouns if len(n) > 1]
count = Counter(filtered_nouns)

ws_keywords = wb.create_sheet("키워드 전체분석")
ws_keywords.append(["키워드", "빈도수"])

for word, freq in sorted(count.items(), key = lambda x: x[1], reverse = True) :
    ws_keywords.append([word, freq])

wb.save("연관키워드_분석결과(청바지구글).xlsx")
print("엑셀저장 완료")

엑셀저장 완료


In [40]:
#데이터 값 설정하기
from openpyxl import load_workbook
from konlpy.tag import Okt
from collections import Counter
import re

wb = load_workbook("연관키워드 20250722 링귀걸이.xlsx")
ws = wb.active

texts = [row[0] for row in ws.iter_rows(min_row=2, max_col=1, values_only=True) if row[0]]

okt = Okt()
all_text = " ".join(texts)

nouns_tokens = okt.nouns(all_text)

regex_tokens = re.findall(r'[A-Za-z]*\d+[A-Za-z]+|\d+[A-Za-z]*', all_text)

valid_one_letter = {"금", "은", "동"}#한글자 중에서도 필요한것
stopwords = {"사용", "정도", "조금", "제품", "때문", "생각", "만족", "구매", "다음"}#불필요한 단어

combined_tokens = nouns_tokens + regex_tokens#
filtered_tokens = [t for t in combined_tokens if (len(t) > 1 or t in valid_one_letter) and t not in stopwords]


count = Counter(filtered_tokens)

ws_keywords = wb.create_sheet("키워드 전체분석")
ws_keywords.append(["키워드", "빈도수"])

for word, freq in sorted(count.items(), key = lambda x: x[1], reverse = True) :
    ws_keywords.append([word, freq])

wb.save("연관키워드_분석결과(링귀걸이v1).xlsx")
print("엑셀저장 완료")

엑셀저장 완료
